In [1]:
!pip install -q datasets

In [2]:
from datasets import load_dataset

ds = load_dataset("Voxel51/WE3DS", split="train", streaming=True)

for i, s in enumerate(ds):
    if i == 0:
        print("FIELDS:")
        for k, v in s.items():
            print(f"  {k:20} {type(v).__name__}  {str(v)[:70]}")
    if i >= 2:
        break

print("\n--- 3rd sample, depth fields ---")
for k in s:
    if "depth" in k.lower():
        print(k, "→", type(s[k]), str(s[k])[:120])

README.md:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/10276 [00:00<?, ?it/s]

FIELDS:
  image                PngImageFile  <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1600x1144 at 0x7C

--- 3rd sample, depth fields ---


In [3]:
from huggingface_hub import list_repo_files

files = list_repo_files("Voxel51/WE3DS", repo_type="dataset")
print("total files:", len(files))

print("\n--- first 30 files ---")
for f in files[:30]:
    print(" ", f)

print("\n--- folders ---")
import collections
dirs = collections.Counter("/".join(f.split("/")[:-1]) for f in files)
for d, n in dirs.most_common(15):
    print(f"  {n:>6}  {d}")

print("\n--- files containing 'depth' ---")
dep = [f for f in files if "depth" in f.lower()]
print("count:", len(dep))
for f in dep[:5]:
    print(" ", f)

total files: 10279

--- first 30 files ---
  .gitattributes
  README.md
  data/img_00000.png
  data/img_00001.png
  data/img_00002.png
  data/img_00003.png
  data/img_00004.png
  data/img_00005.png
  data/img_00006.png
  data/img_00007.png
  data/img_00008.png
  data/img_00009.png
  data/img_00010.png
  data/img_00011.png
  data/img_00012.png
  data/img_00013.png
  data/img_00014.png
  data/img_00015.png
  data/img_00016.png
  data/img_00017.png
  data/img_00018.png
  data/img_00019.png
  data/img_00020.png
  data/img_00021.png
  data/img_00022.png
  data/img_00023.png
  data/img_00024.png
  data/img_00025.png
  data/img_00026.png
  data/img_00027.png

--- folders ---
    2568  data
    2568  fields/depth_raw
    2568  fields/depth_refined
    2568  fields/ground_truth
       7  

--- files containing 'depth' ---
count: 5136
  fields/depth_raw/img_00000.png
  fields/depth_raw/img_00001.png
  fields/depth_raw/img_00002.png
  fields/depth_raw/img_00003.png
  fields/depth_raw/img_00004.pn

In [4]:
from huggingface_hub import hf_hub_download
from PIL import Image
import numpy as np

REPO = "Voxel51/WE3DS"

for idx in ["img_00000", "img_00007"]:
    print("=" * 60)
    for sub in ["data", "fields/depth_raw", "fields/depth_refined"]:
        p = hf_hub_download(REPO, f"{sub}/{idx}.png", repo_type="dataset")
        im = Image.open(p)
        a = np.array(im)
        print(f"{sub:24} mode={im.mode:6} shape={a.shape} dtype={a.dtype} "
              f"min={a.min()} max={a.max()}")

data/img_00000.png: reconstructing file:   0%|          |  0.00B / 3.04MB            

data/img_00000.png: downloading bytes:           |  0.00B            

data                     mode=RGB    shape=(1144, 1600, 3) dtype=uint8 min=3 max=255


fields/depth_raw/img_00000.png: reconstructing file:   0%|          |  0.00B /  758kB            

fields/depth_raw/img_00000.png: downloading bytes:           |  0.00B            

...

fields/depth_raw/img_00007.png: downloading bytes:           |  0.00B            

fields/depth_raw         mode=I;16   shape=(1144, 1600) dtype=uint16 min=0 max=7145


fields/depth_refined/img_00007.png: reconstructing file:   0%|          |  0.00B /  770kB            

fields/depth_refined/img_00007.png: downloading bytes:           |  0.00B            

fields/depth_refined     mode=I;16   shape=(1144, 1600) dtype=uint16 min=5566 max=7145


In [5]:
# ============================================================
# STEP 1 — download the subset once, in parallel
# ============================================================
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # needs: pip install hf_transfer

from huggingface_hub import snapshot_download

N_IMAGES = 80
LOCAL    = "/kaggle/working/we3ds"

patterns = []
for i in range(N_IMAGES):
    patterns += [f"data/img_{i:05d}.png",
                 f"fields/depth_refined/img_{i:05d}.png"]

snapshot_download(
    "Voxel51/WE3DS",
    repo_type="dataset",
    allow_patterns=patterns,
    local_dir=LOCAL,
    max_workers=8,
)
print("downloaded to", LOCAL)

# ============================================================
# STEP 2 — Depth Anything V2 vs measured depth
# ============================================================
import numpy as np, pandas as pd, torch, glob
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from scipy.stats import spearmanr

MODEL     = "depth-anything/Depth-Anything-V2-Small-hf"
GRID      = 8      # 8 x 8 = 64 patches per image
MIN_VALID = 0.5    # a patch is used only if >50% of its GT pixels are valid

processor = AutoImageProcessor.from_pretrained(MODEL)
model = AutoModelForDepthEstimation.from_pretrained(MODEL).eval()

rows, per_image = [], []
names = sorted(os.path.basename(p) for p in glob.glob(f"{LOCAL}/data/*.png"))
print(f"{len(names)} image pairs found\n")

for name in names:
    rgb = Image.open(f"{LOCAL}/data/{name}").convert("RGB")
    gt  = np.array(Image.open(f"{LOCAL}/fields/depth_refined/{name}")).astype(np.float32)
    H, W = gt.shape

    # Depth Anything V2 — identical to the detection pipeline
    inputs = processor(images=rgb, return_tensors="pt")
    with torch.no_grad():
        pred = model(**inputs).predicted_depth
    pred = torch.nn.functional.interpolate(
        pred.unsqueeze(1), size=(H, W), mode="bicubic", align_corners=False
    )[0, 0].numpy()
    lo, hi = float(np.nanmin(pred)), float(np.nanmax(pred))
    z_pred = (pred - lo) / (hi - lo) if hi > lo else np.zeros_like(pred)

    # ROI medians on both sides, exactly as Z_rel is computed in the paper
    hs, ws = H // GRID, W // GRID
    zp, zg = [], []
    for r in range(GRID):
        for c in range(GRID):
            sl = (slice(r * hs, (r + 1) * hs), slice(c * ws, (c + 1) * ws))
            g = gt[sl]; m = g > 0
            if m.sum() < MIN_VALID * g.size:
                continue
            zp.append(float(np.median(z_pred[sl][m])))
            zg.append(float(np.median(g[m])))

    if len(zp) < 10:
        print(f"{name}  too few valid patches"); continue

    rho, _ = spearmanr(zp, zg)
    per_image.append(rho)
    rows += [{"image": name, "z_pred": a, "gt_mm": b} for a, b in zip(zp, zg)]
    print(f"{name}  patches={len(zp):>3}  rho={rho:+.3f}")

pi = np.array(per_image)
pd.DataFrame(rows).to_csv("/kaggle/working/we3ds_depth_validation.csv", index=False)

print("\n" + "=" * 64)
print(f"images used            : {len(pi)}")
print(f"ROI pairs              : {len(rows)}")
print(f"per-image Spearman rho : mean {pi.mean():+.3f}   median {np.median(pi):+.3f}")
print(f"                         IQR [{np.percentile(pi,25):+.3f}, {np.percentile(pi,75):+.3f}]")
print(f"negative rho in        : {100*(pi<0).mean():.0f}% of images")
print(f"|rho| > 0.5 in         : {100*(np.abs(pi)>0.5).mean():.0f}% of images")
print("=" * 64)
print("GT is measured distance (mm): larger = farther.")
print("A strong NEGATIVE rho means the model's normalized output is larger for")
print("NEARER surfaces, i.e. it behaves as relative inverse depth.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 160 files:   0%|          | 0/160 [00:00<?, ?it/s]

downloaded to /kaggle/working/we3ds


preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 99.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

80 image pairs found

img_00000.png  patches= 64  rho=-0.661
img_00001.png  patches= 64  rho=-0.760
img_00002.png  patches= 64  rho=-0.830
img_00003.png  patches= 64  rho=-0.697
img_00004.png  patches= 64  rho=-0.756
img_00005.png  patches= 64  rho=-0.656
img_00006.png  patches= 64  rho=-0.703
img_00007.png  patches= 64  rho=-0.735
img_00008.png  patches= 64  rho=-0.399
img_00009.png  patches= 64  rho=-0.482
img_00010.png  patches= 64  rho=-0.606
img_00011.png  patches= 64  rho=-0.335
img_00012.png  patches= 64  rho=-0.255
img_00013.png  patches= 64  rho=-0.619
img_00014.png  patches= 64  rho=-0.621
img_00015.png  patches= 64  rho=-0.500
img_00016.png  patches= 64  rho=-0.590
img_00017.png  patches= 64  rho=-0.648
img_00018.png  patches= 64  rho=-0.822
img_00019.png  patches= 64  rho=-0.568
img_00020.png  patches= 64  rho=-0.572
img_00021.png  patches= 64  rho=-0.582
img_00022.png  patches= 64  rho=-0.249
img_00023.png  patches= 64  rho=-0.457
img_00024.png  patches= 64  rho=-0.469
img